In [1]:
%pip install --quiet -U langgraph typing_extensions langchain_openrouter dotenv exa_py

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [6]:
from typing import TypedDict
from pydantic import BaseModel

class AnalysisState(BaseModel):
    core_question: str
    core_claim: str
    assumptions: list[str]
    objections: list[str]
    alternative_views: list[str]
    uncertainties: list[str]

class WritingState(BaseModel):
    introduction: str
    body: str
    conclusion: str

class GraphState(TypedDict, total=False):
    question: str
    analysis: AnalysisState
    writing: WritingState

In [ ]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0
)

analysis_model = model.with_structured_output(
    AnalysisState
)

writing_model = model.with_structured_output(
    WritingState
)


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage


def analyse_question(state: GraphState) -> dict:
    system_message = SystemMessage(
        content="""
You are a critical thinking assistant.

Analyse the user's question and identify:

- core_question: 
- core_claim: 
- assumptions: 
- objections: 
- alternative_views: 
- uncertainties:

Do not write a polished final answer.
Do not invent evidence.
Return only the requested structured output.
"""
    )

    human_message = HumanMessage(
        content=state["question"]
    )


    result = analysis_model.invoke(
        [system_message, human_message]
    )

    return {
        "analysis": result
    })

def write_answer(state: GraphState) -> dict:
    analysis = state["analysis"]

    system_message = SystemMessage(
        content="""Write a well structured essay based on the analysis provided.""")
    
    user_msg = HumanMessage(
        content=f"""
Based on the following analysis, write a well structured essay. Body can containt multiple paragraphs.
{analysis.model_dump_json(indent=2)}
""" 
    )
    result = writing_model.invoke(
        [system_message, user_msg]
    )
    return {"writing": result}

In [33]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(GraphState)

builder.add_node("analyse_question", analyse_question)
builder.add_node("write_answer", write_answer)

builder.add_edge(START, "analyse_question")
builder.add_edge("analyse_question", "write_answer")
builder.add_edge("write_answer", END)

graph = builder.compile()

In [36]:
result = graph.invoke(
    {"question": "What should I read to deepen my understanding of information technology?"}
)

result

{'question': 'What should I read to deepen my understanding of information technology?',
 'analysis': AnalysisState(core_question="What reading materials would best deepen one's understanding of information technology?", core_claim="There exists a set of reading materials that can effectively deepen one's understanding of IT.", assumptions=['The user has some baseline knowledge (but level is unknown)', 'Information technology is a coherent field with definable core texts', 'Reading is an effective primary method for deepening understanding (vs hands-on practice, courses, etc.)', 'The user wants breadth across IT vs depth in a specialization (unclear)', "The user's goals are general understanding vs career preparation (unclear)"], objections=['IT is too broad for a single reading list to be meaningful', 'Books become outdated quickly in fast-moving IT subfields', 'Practical experience and hands-on labs matter more than reading for skill development', 'Different subfields (security, devo

In [37]:
print(result["writing"].introduction)

The question of what reading materials best deepen one's understanding of information technology (IT) is deceptively simple. While there is a widespread belief that a canonical set of books or texts can serve as a universal gateway to IT mastery, the reality is far more nuanced. The field of information technology is vast, rapidly evolving, and encompasses diverse sub-disciplines—from cybersecurity and cloud computing to software development and data engineering. This essay examines the assumptions underlying the search for a definitive IT reading list, explores the objections that challenge its feasibility, considers alternative learning paradigms, and highlights the critical uncertainties that must be addressed before any meaningful recommendation can be made.


In [38]:
from IPython.display import display, Markdown

writing = result["writing"]

essay = f"""
# Introduction

{writing.introduction}

# Body

{writing.body}

# Conclusion

{writing.conclusion}
"""

display(Markdown(essay))


# Introduction

The question of what reading materials best deepen one's understanding of information technology (IT) is deceptively simple. While there is a widespread belief that a canonical set of books or texts can serve as a universal gateway to IT mastery, the reality is far more nuanced. The field of information technology is vast, rapidly evolving, and encompasses diverse sub-disciplines—from cybersecurity and cloud computing to software development and data engineering. This essay examines the assumptions underlying the search for a definitive IT reading list, explores the objections that challenge its feasibility, considers alternative learning paradigms, and highlights the critical uncertainties that must be addressed before any meaningful recommendation can be made.

# Body

The core claim—that a set of reading materials exists to effectively deepen IT understanding—rests on several implicit assumptions. First, it assumes the user possesses some baseline knowledge, though the level remains unspecified. Second, it treats IT as a coherent field with definable core texts, akin to the great books of literature or philosophy. Third, it privileges reading as the primary mode of deep learning, potentially undervaluing hands-on practice, interactive labs, or structured courses. Fourth, it assumes the user seeks breadth across IT rather than depth in a specialization, and that their goals are general understanding rather than targeted career preparation. These assumptions, while reasonable as starting points, risk oversimplifying a landscape where context is everything.  Objections to a universal IT reading list are substantial and multifaceted. The breadth of IT makes any single list either superficial or unwieldy. Books, especially in fast-moving subfields like cloud architecture or AI/ML, can become outdated before they reach print. Practical experience—configuring networks, writing code, responding to incidents—often yields deeper, more durable understanding than passive reading. Moreover, the reading list for a network engineer differs fundamentally from that of a data scientist or a security analyst. Learning styles further complicate the picture: many learners thrive on interactive platforms, video tutorials, or project-based curricula rather than traditional texts.  Alternative views offer constructive pathways forward. Some argue for grounding oneself first in foundational computer science—algorithms, data structures, operating systems, and distributed systems—before tackling IT-specific topics. Curated open-source curricula like OSSU (Open Source Society University) or TeachYourselfCS provide structured, peer-reviewed roadmaps that blend theory and practice. Project-based learning, where one builds real systems, is often more effective than reading alone. Early specialization in a subfield—cybersecurity, cloud, data engineering, or software development—allows for targeted, relevant resource selection. Vendor certification paths (AWS, Cisco, CompTIA, Microsoft) offer another structured framework, aligning learning with industry-recognized benchmarks.  Ultimately, the uncertainties surrounding the user's profile render any generic recommendation speculative. We do not know their current knowledge level, specific interests, learning goals, preferred format, time availability, or whether they prioritize theoretical foundations or immediately applicable skills. Without this context, a reading list is at best a shot in the dark and at worst a source of frustration and wasted effort.

# Conclusion

In conclusion, while the desire for a definitive IT reading list is understandable, the search for one is likely misguided. Information technology is not a monolith but a federation of rapidly evolving disciplines, each with its own foundational texts, practical demands, and learning cultures. The most effective approach is not to seek a universal canon but to first clarify one's goals, background, and learning preferences. From there, a personalized curriculum—blending foundational computer science, targeted subfield resources, hands-on projects, and perhaps vendor-aligned certifications—will yield far deeper and more durable understanding than any static list of books. The best reading material for IT is not a fixed set of titles, but a dynamic, context-aware strategy that evolves with the learner and the field.


In [39]:
from IPython.display import display, Markdown
analysis = result["analysis"]
display(
    Markdown(
        f"```json\n{analysis.model_dump_json(indent=2)}\n```"
    )
)

```json
{
  "core_question": "What reading materials would best deepen one's understanding of information technology?",
  "core_claim": "There exists a set of reading materials that can effectively deepen one's understanding of IT.",
  "assumptions": [
    "The user has some baseline knowledge (but level is unknown)",
    "Information technology is a coherent field with definable core texts",
    "Reading is an effective primary method for deepening understanding (vs hands-on practice, courses, etc.)",
    "The user wants breadth across IT vs depth in a specialization (unclear)",
    "The user's goals are general understanding vs career preparation (unclear)"
  ],
  "objections": [
    "IT is too broad for a single reading list to be meaningful",
    "Books become outdated quickly in fast-moving IT subfields",
    "Practical experience and hands-on labs matter more than reading for skill development",
    "Different subfields (security, devops, data, networking, development) require completely different reading lists",
    "Learning styles vary - some prefer interactive platforms, videos, or structured courses over books"
  ],
  "alternative_views": [
    "Focus first on foundational computer science concepts (algorithms, data structures, systems) before IT-specific topics",
    "Follow curated open-source curricula like OSSU (Open Source Society University) or TeachYourselfCS",
    "Prioritize project-based learning and building things over passive reading",
    "Specialize early in a subfield (cybersecurity, cloud, data engineering, software development) with targeted resources",
    "Use vendor certification study paths (AWS, Cisco, CompTIA, Microsoft) as structured learning frameworks"
  ],
  "uncertainties": [
    "User's current knowledge level (beginner, intermediate, advanced)",
    "User's specific interests within IT (infrastructure, development, security, data, management)",
    "User's learning goals (career transition, advancement, academic interest, hobby)",
    "Preferred learning format (books, interactive platforms, video courses, documentation)",
    "Time commitment available for study",
    "Whether they want theoretical foundations or immediately applicable practical skills"
  ]
}
```